In [1]:
import os
from getpass import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "learning"

In [32]:
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os
load_dotenv()

creative_llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    temperature=0.9
)

llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    temperature=0.0
)
llm1 = ChatOpenRouter(
    model = "openai/gpt-oss-20b"
)
print(llm.invoke("What is the capital of India?").content)
print(llm1.invoke("What is the capital of India?").content)

New Delhi.
The capital of India is New Delhi.


In [3]:
prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Context: {context}
"""



LangChain uses a ChatPromptTemplate object to format the various prompt types into a single list which will be passed to our LLM:

In [5]:
from langchain_classic.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", prompt),
    ("user","{query}")
])

In [6]:
prompt_template.input_variables

['context', 'query']

In [7]:
prompt_template.messages


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]

In [8]:
from langchain_classic.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(prompt),
    HumanMessagePromptTemplate.from_template("{query}"),
])

In [9]:

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]

In [12]:
pipeline = (
    {
        "query" : lambda x: x["query"],
        "context": lambda x: x["context"]
    }
    | prompt_template | llm1
)

In [13]:
context = """Aurelio AI is an AI company developing tooling for AI
engineers. Their focus is on language AI with the team having strong
expertise in building AI agents and a strong background in
information retrieval.

The company is behind several open source frameworks, most notably
Semantic Router and Semantic Chunkers. They also have an AI
Platform providing engineers with tooling to help them build with
AI. Finally, the team also provides development services to other
organizations to help them bring their AI tech to market.

Aurelio AI became LangChain Experts in September 2024 after a long
track record of delivering AI solutions built with the LangChain
ecosystem."""

query = "What is Aurelio AI's expertise?"

In [14]:
pipeline.invoke({
    "query": query, 
    "context" : context
})

AIMessage(content='Aurelio AI specializes in language AI and tooling for AI engineers, with particular strengths in building AI agents and information retrieval. They develop open-source frameworks (notably Semantic Router and Semantic Chunkers), run an AI Platform for engineers, and offer development services to help organizations bring AI products to market. They also became LangChain Experts in September 2024.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 183, 'total_tokens': 265, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DfNsNNWmE681SuLC7rRqt2IpJvdUM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019

In [15]:
result = pipeline.invoke({
    "query": query, 
    "context" : context
})
print(result.content)

Aurelio AI specializes in language AI for engineers, with particular expertise in building AI agents and information retrieval. They develop open-source frameworks (notably Semantic Router and Semantic Chunkers), provide an AI Platform with tooling for AI engineering, and offer development services to help organizations bring AI products to market. They also have a track record in the LangChain ecosystem, becoming LangChain Experts in September 2024.


### Few Shot Prompting

In [16]:
e_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

In [17]:
examples = [
    {"input": "Here is query #1", "output": "Here is the answer #1"},
    {"input": "Here is query #2", "output": "Here is the answer #2"},
    {"input": "Here is query #3", "output": "Here is the answer #3"},
]


In [19]:
from langchain_classic.prompts import FewShotChatMessagePromptTemplate

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=e_prompt,
    examples=examples,)

print(few_shot_prompt.format())
    

Human: Here is query #1
AI: Here is the answer #1
Human: Here is query #2
AI: Here is the answer #2
Human: Here is query #3
AI: Here is the answer #3


In [20]:
new_system_prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Always answer in markdown format. When doing so please
provide headers, short summaries, follow with bullet
points, then conclude.

Context: {context}
"""

In [21]:
prompt_template.messages[0].prompt.template = new_system_prompt
out = pipeline.invoke({"query": query, "context": context}).content
print(out)

## Summary
Aurelio AI specializes in building tooling and solutions for language AI, with deep expertise in AI agents and information retrieval.

- Focus: language AI
- Strong skills: building AI agents
- Strong background: information retrieval

## Products and contributions
- Open-source frameworks: Semantic Router, Semantic Chunkers
- AI Platform: tooling for AI engineers to build with AI
- Services: development services to help organizations bring AI tech to market

## Recognition
- Named LangChain Experts in September 2024 for their track record delivering AI solutions in the LangChain ecosystem

Conclusion: Aurelio AI’s expertise centers on language AI, especially agent-based systems and information retrieval, supported by open-source tools, a platform, and development services.


In [22]:
from IPython.display import display, Markdown

display(Markdown(out))



## Summary
Aurelio AI specializes in building tooling and solutions for language AI, with deep expertise in AI agents and information retrieval.

- Focus: language AI
- Strong skills: building AI agents
- Strong background: information retrieval

## Products and contributions
- Open-source frameworks: Semantic Router, Semantic Chunkers
- AI Platform: tooling for AI engineers to build with AI
- Services: development services to help organizations bring AI tech to market

## Recognition
- Named LangChain Experts in September 2024 for their track record delivering AI solutions in the LangChain ecosystem

Conclusion: Aurelio AI’s expertise centers on language AI, especially agent-based systems and information retrieval, supported by open-source tools, a platform, and development services.

In [23]:
examples = [
    {
        "input": "Can you explain gravity?",
        "output": (
            "## Gravity\n\n"
            "Gravity is one of the fundamental forces in the universe.\n\n"
            "### Discovery\n\n"
            "* Gravity was first discovered by Sir Isaac Newton in the late 17th century.\n"
            "* It was said that Newton theorized about gravity after seeing an apple fall from a tree.\n\n"
            "### In General Relativity\n\n"
            "* Gravity is described as the curvature of spacetime.\n"
            "* The more massive an object is, the more it curves spacetime.\n"
            "* This curvature is what causes objects to fall towards each other.\n\n"
            "### Gravitons\n\n"
            "* Gravitons are hypothetical particles that mediate the force of gravity.\n"
            "* They have not yet been detected.\n\n"
            "**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.\n\n"
        )
    },
    {
        "input": "What is the capital of France?",
        "output": (
            "## France\n\n"
            "The capital of France is Paris.\n\n"
            "### Origins\n\n"
            "* The name Paris comes from the Latin word \"Parisini\" which referred to a Celtic people living in the area.\n"
            "* The Romans named the city Lutetia, which means \"the place where the river turns\".\n"
            "* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.\n\n"
            "**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.\n\n"
        )
    }
]

In [24]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=e_prompt,
    examples=examples,)

In [25]:
out = few_shot_prompt.format()

display(Markdown(out))



Human: Can you explain gravity?
AI: ## Gravity

Gravity is one of the fundamental forces in the universe.

### Discovery

* Gravity was first discovered by Sir Isaac Newton in the late 17th century.
* It was said that Newton theorized about gravity after seeing an apple fall from a tree.

### In General Relativity

* Gravity is described as the curvature of spacetime.
* The more massive an object is, the more it curves spacetime.
* This curvature is what causes objects to fall towards each other.

### Gravitons

* Gravitons are hypothetical particles that mediate the force of gravity.
* They have not yet been detected.

**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.


Human: What is the capital of France?
AI: ## France

The capital of France is Paris.

### Origins

* The name Paris comes from the Latin word "Parisini" which referred to a Celtic people living in the area.
* The Romans named the city Lutetia, which means "the place where the river turns".
* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.

**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.



In [26]:

few_shot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'Can you explain gravity?', 'output': '## Gravity\n\nGravity is one of the fundamental forces in the universe.\n\n### Discovery\n\n* Gravity was first discovered by Sir Isaac Newton in the late 17th century.\n* It was said that Newton theorized about gravity after seeing an apple fall from a tree.\n\n### In General Relativity\n\n* Gravity is described as the curvature of spacetime.\n* The more massive an object is, the more it curves spacetime.\n* This curvature is what causes objects to fall towards each other.\n\n### Gravitons\n\n* Gravitons are hypothetical particles that mediate the force of gravity.\n* They have not yet been detected.\n\n**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.\n\n'}, {'input': 'What is the capital of France?', 'output': '## France\n\nThe capital of France is Paris.\n\n### Origins\n\n* The name Paris comes from the Latin word "Parisini" which refe

In [27]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", new_system_prompt),
    few_shot_prompt,
    ("user", "{query}"),
])

In [28]:
pipeline = prompt_template | llm
out = pipeline.invoke({"query": query, "context": context}).content
display(Markdown(out))


## Aurelio AI’s Areas of Expertise

Aurelio AI specializes in building and optimizing language‑model‑centric tooling for AI engineers. Their key strengths include:

- **Language‑AI Engineering**  
  - Expertise in developing applications that leverage large language models (LLMs) to perform natural‑language tasks.
  - Skilled at training, fine‑tuning, and deploying such models for commercial use.

- **AI Agents & Retrieval**  
  - Strong background in building intelligent agents that can autonomously perform complex tasks.
  - Advanced knowledge in information retrieval systems that enhance LLM performance.

- **Open‑Source Frameworks**  
  - *Semantic Router*: A framework for directing user queries to the most suitable LLM or knowledge base.
  - *Semantic Chunkers*: Tools that slice large texts into semantically meaningful fragments for efficient processing.

- **AI Platform & Tooling**  
  - Architected an AI platform that offers end‑to‑end pipelines, monitoring, and reproducibility for AI projects.
  - Provides APIs, SDKs, and documentation that streamline the workflow for engineering teams.

- **Custom Development Services**  
  - Builds tailored AI solutions for external partners, helping them bring complex AI capabilities to market quickly.

- **LangChain Expertise**  
  - Recognized as LangChain Experts (since September 2024) due to extensive experience delivering solutions built with the LangChain ecosystem.

> **Bottom line** – Aurelio AI excels at turning advanced language‑model technology into practical, production‑ready tools and services for AI developers and enterprises.

### Chain of Thought Prompting

In [29]:
no_cot_system_prompt = """
Be a helpful assistant and answer the user's question.

You MUST answer the question directly without any other
text or explanation.
"""

no_cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", no_cot_system_prompt),
    ("user", "{query}"),
])



In [33]:
query = (
    "How many keystrokes are needed to type the numbers from 1 to 500?"
)

no_cot_pipeline = no_cot_prompt_template | llm1
no_cot_result = no_cot_pipeline.invoke({"query": query}).content
print(no_cot_result)



1392


In [34]:
cot_system_prompt = """
Be a helpful assistant and answer the user's question.

To answer the question, you must:

- List systematically and in precise detail all
  subproblems that need to be solved to answer the
  question.
- Solve each sub problem INDIVIDUALLY and in sequence.
- Finally, use everything you have worked through to
  provide the final answer.
"""

In [35]:
cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", cot_system_prompt),
    ("user", "{query}"),
])

cot_pipeline = cot_prompt_template | llm1

In [36]:
cot_result =  cot_pipeline.invoke({"query": query}).content
display(Markdown(cot_result))


**Sub‑problems**

1. **Identify the different ranges of numbers (by number of digits).**  
2. **Count how many numbers fall into each range.**  
3. **Compute the total number of digits (keystrokes) for each range.**  
4. **Sum the counts from all ranges.**  
5. **Decide whether to include separators (spaces, new‑lines).**

---

### 1. Groups by length

| Length of number | Numbers in this group | Count of numbers |
|------------------|-----------------------|------------------|
| 1 digit          | 1 – 9                 | 9 |
| 2 digits         | 10 – 99               | 90 |
| 3 digits         | 100 – 499             | 400 |
| 3 digits (500)   | 500                   | 1 |

(Only one 3‑digit number beyond 499.)

---

### 2. Keystrokes per group

- **1‑digit numbers:** \(9 \text{ numbers} \times 1 \text{ digit} = 9\) keystrokes  
- **2‑digit numbers:** \(90 \times 2 = 180\) keystrokes  
- **3‑digit numbers 100–499:** \(400 \times 3 = 1\,200\) keystrokes  
- **Number 500:** \(1 \times 3 = 3\) keystrokes  

(If we were to type a space after each number except the last, that would add 499 more keystrokes; we’ll first compute without separators.)

---

### 3. Total keystrokes (digits only)

\[
9 + 180 + 1\,200 + 3 = 1\,392 \text{ keystrokes}
\]

---

### 4. (Optional) Including separators

If you press a space after every number except the final one (500), add 499 keystrokes:

\[
1\,392 + 499 = 1\,891 \text{ keystrokes}
\]

---

## Final answer

- **Keystrokes to type the digits 1–500:** **1 392**  
- **Keystrokes if you also count the spaces between numbers:** **1 891**

Most interpretations of the question require the first figure, so **1392 keystrokes** is the typical answer.

In [37]:
system_prompt = """
Be a helpful assistant and answer the user's question.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{query}"),
])

pipeline = prompt_template | llm



In [38]:


result = pipeline.invoke({"query": query}).content
display(Markdown(result))



We want the total number of digits (keystrokes) needed to type all the integers from 1 through 500 inclusive.

Count by digit-length:

- 1-digit numbers: 1 through 9 → 9 numbers × 1 digit = 9 digits
- 2-digit numbers: 10 through 99 → 90 numbers × 2 digits = 180 digits
- 3-digit numbers: 100 through 499 → 400 numbers × 3 digits = 1200 digits
- 500: a single 3-digit number → 1 number × 3 digits = 3 digits

Add them: 9 + 180 + 1200 + 3 = 1392

So 1,392 keystrokes are needed.

CoT is useful not only for simple question-answering like this, but is also a fundamental component of many agentic systems which will often use CoT steps paired with tool use to solve very complex problems, this is what we see in OpenAI's current flagship model o1. We'll see later in the course how we can do this ourselves.